# Day 3 — Advanced RAG Pipeline

Production-grade RAG: Semantic Cache + Hybrid Retrieval + Reranking + LangSmith Tracing.
Every component from Day 3 combined into one end-to-end pipeline that handles real-world scale.
This is the architecture used by teams at JP Morgan, Shell, and LTIMindtree for enterprise knowledge retrieval.

In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np
from day3.rag_pipeline import (
    SemanticCache, LangSmithTracer, AdvancedRAGPipeline,
    AdvancedRAGResult, build_product_rag, mock_claude_response
)
from day3.hybrid_search import generate_product_corpus

def mock_embed(texts):
    vecs = []
    for t in texts:
        rng = np.random.default_rng(abs(hash(t)) % (2**31))
        v = rng.standard_normal(384).astype(np.float32)
        v = v / np.linalg.norm(v)
        vecs.append(v)
    return np.array(vecs)

print("Advanced RAG Pipeline loaded")

## 1. Semantic Cache

Traditional caches need exact query matches. Semantic cache matches by MEANING.
'What is the MacBook price?' returns the cached answer for 'How much does the MacBook cost?'.
Threshold 0.92 = only near-identical rephrasing hits the cache.

In [ ]:
cache = SemanticCache(similarity_threshold=0.92, max_size=100)

# Prime the cache
cache.put("What is the price of MacBook Pro?", "MacBook Pro 14 M3 Pro costs \u20b91,99,900.")
cache.put("Which headphones have ANC?", "Sony WH-1000XM5 has industry-leading ANC with 8 mics.")

# Test cache hits
test_queries = [
    ("What is the price of MacBook Pro?",     "exact match"),
    ("How much does the MacBook Pro cost?",   "paraphrase \u2014 should MISS at 0.92"),
    ("Which headphones have ANC?",            "exact match"),
    ("Best headphones for noise cancellation?", "paraphrase \u2014 should MISS at 0.92"),
]

for query, note in test_queries:
    result = cache.get(query)
    hit = "HIT \u2713" if result else "MISS \u2717"
    print(f"  [{hit}] '{query[:55]}'")
    if result:
        print(f"         \u2192 '{result[:80]}'")

print(f"\nCache stats: {cache.stats()}")

## 2. LangSmith Tracer

LangSmith records every LLM call with query, context, answer, latency, and metadata.
Without LANGCHAIN_API_KEY it falls back to local JSON.
This gives you full observability even in development.

In [ ]:
import os
tracer = LangSmithTracer(project_name="day3-advanced-rag")

# Log some mock traces
trace_id = tracer.log_rag_call(
    query       = "What is the best laptop for ML?",
    context     = "MacBook Pro 14 M3 Pro with 18GB unified memory...",
    answer      = "MacBook Pro 14 M3 Pro is best for ML with M3 Pro chip and 18GB memory.",
    latency_ms  = 245.3,
    metadata    = {"method": "hybrid_rrf", "num_retrieved": 10, "cache_hit": False},
)
print(f"Trace logged: {trace_id}")

traces = tracer.get_traces()
print(f"Total traces: {len(traces)}")
print(f"Latest: {traces[-1]}")

## 3. Building the Pipeline

AdvancedRAGPipeline combines all Day 3 components.
Pass mock_embed_fn to avoid model downloads.
In production, remove mock_embed_fn and the pipeline uses all-MiniLM-L6-v2.

In [ ]:
pipeline = build_product_rag(mock_embed_fn=mock_embed)
print("Pipeline built with 20 product documents")
print(f"Config: hybrid=True, reranker=True, cache=True, top_k=10, rerank_top_k=3")

## 4. Querying the Pipeline

Ask questions in natural language. The pipeline runs hybrid retrieval, reranks, calls the LLM (or mock), caches the answer,
and returns a structured AdvancedRAGResult with full metadata.

In [ ]:
questions = [
    "What is the best laptop for machine learning?",
    "Which headphones have the best noise cancellation?",
    "Tell me about the MacBook Pro price and specs.",
]

for question in questions:
    result = pipeline.query(question)
    print(f"\nQ: {question}")
    print(f"A: {result.answer[:150]}")
    print(f"   Method: {result.retrieval_method} | "
          f"Retrieved: {result.num_retrieved} \u2192 Reranked: {result.num_reranked} | "
          f"Cache: {'HIT' if result.cache_hit else 'MISS'} | "
          f"Latency: {result.latency_ms:.1f}ms | "
          f"Faithfulness: {result.faithfulness_proxy:.3f}")

## 5. Semantic Cache in Action

Ask the same question twice — the second call should be a cache hit with near-zero latency.
This demonstrates the 100x speed improvement from semantic caching.

In [ ]:
print("First call (cache MISS \u2014 will run full pipeline):")
result1 = pipeline.query("What is the best laptop for machine learning?")
print(f"  Latency: {result1.latency_ms:.1f}ms | Cache: {'HIT' if result1.cache_hit else 'MISS'}")

print("\nSecond call (cache HIT \u2014 instant return):")
result2 = pipeline.query("What is the best laptop for machine learning?")
print(f"  Latency: {result2.latency_ms:.1f}ms | Cache: {'HIT' if result2.cache_hit else 'MISS'}")

print(f"\nCache stats: {pipeline.cache_stats}")

## 6. Metadata-Filtered Queries

Restrict retrieval to specific categories before running the pipeline.
'Best headphones' filtered to category=headphones gives higher precision.

In [ ]:
print("Without filter:")
r_no_filter = pipeline.query("What should I buy for music?", metadata_filters={})
print(f"  Retrieved: {r_no_filter.num_retrieved} chunks | {r_no_filter.answer[:100]}")

print("\nWith category filter (headphones only):")
r_filtered = pipeline.query(
    "What should I buy for music?",
    metadata_filters={"dominant_category": "headphones"}
)
print(f"  Retrieved: {r_filtered.num_retrieved} chunks | {r_filtered.answer[:100]}")

## 7. Save Traces

Save all LangSmith traces to a local JSON file for review.
In production, view these at smith.langchain.com with a LANGCHAIN_API_KEY.

In [ ]:
pipeline._tracer.save_traces("traces/notebook_traces.json")
all_traces = pipeline._tracer.get_traces()
print(f"Saved {len(all_traces)} traces")
if all_traces:
    latest = all_traces[-1]
    print(f"Latest trace: {latest['trace_id']} | latency={latest['latency_ms']}ms | method={latest['metadata'].get('method', 'N/A')}")